# 02 · DataFrames en PySpark

**Objetivos**
- Leer CSV y JSON definiendo o infiriendo el esquema.
- Seleccionar, filtrar, crear columnas y agregar datos.
- Tratar valores nulos y erróneos.
- Unir (join) DataFrames y ver el plan de ejecución.
- Llevar un resultado pequeño a pandas para graficarlo.

> Duración estimada: 30 min

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Fuera de Docker -> local[*]. Dentro del docker-compose -> spark://spark-master:7077
MASTER = os.environ.get("SPARK_MASTER", "local[*]")

builder = (SparkSession.builder
           .appName("02_DataFrames")
           .master(MASTER)
           .config("spark.sql.shuffle.partitions", "8"))   # pocos datos -> pocas particiones
if os.environ.get("SPARK_DRIVER_HOST"):
    builder = builder.config("spark.driver.host", os.environ["SPARK_DRIVER_HOST"])

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

DATA = os.path.abspath("../data")   # carpeta de datos del repositorio
print("Spark", spark.version, "| master:", sc.master)
print("Spark UI:", sc.uiWebUrl)

## 1. Lectura de datos

In [ ]:
ventas = spark.read.csv(f"{DATA}/ventas.csv", header=True, inferSchema=True)
clientes = spark.read.csv(f"{DATA}/clientes.csv", header=True, inferSchema=True)
productos = spark.read.json(f"{DATA}/productos.json")   # JSON Lines (un objeto por línea)

ventas.printSchema()
print("Filas de ventas:", ventas.count())

In [ ]:
ventas.show(5)

In [ ]:
productos.show(5, truncate=False)

> `inferSchema=True` recorre los datos para deducir los tipos. En entornos productivos es preferible **definir el esquema** de forma explícita, por ser más rápido y seguro (The Apache Software Foundation [ASF], 2026b); así se hace en `src/etl_ventas.py`.

## 2. Explorar

In [ ]:
ventas.describe("cantidad", "precio_unitario", "descuento").show()

In [ ]:
ventas.select("canal").distinct().show()

## 3. Seleccionar, filtrar y crear columnas

In [ ]:
(ventas
 .select("id_venta", "fecha", "cantidad", "precio_unitario", "canal")
 .filter((F.col("canal") == "Web") & (F.col("cantidad") >= 3))
 .orderBy(F.desc("precio_unitario"))
 .show(5))

In [ ]:
ventas2 = (ventas
           .withColumn("total", F.col("cantidad") * F.col("precio_unitario") * (1 - F.col("descuento")))
           .withColumn("mes", F.month("fecha"))
           .withColumn("dia_semana", F.date_format("fecha", "E"))
           .withColumnRenamed("metodo_pago", "pago"))
ventas2.select("id_venta", "fecha", "mes", "dia_semana", "total", "pago").show(5)

## 4. Calidad de datos: nulos y valores inválidos
El conjunto de datos incluye, de forma intencional, cerca de un 1 % de registros con errores.

In [ ]:
# Contar nulos por columna
ventas.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ventas.columns]).show()

print("Cantidades negativas:", ventas.filter(F.col("cantidad") < 0).count())

In [ ]:
limpias = (ventas2
           .filter(F.col("cantidad").isNotNull() & (F.col("cantidad") > 0))
           .fillna({"canal": "Desconocido"}))
print("Antes:", ventas2.count(), "| Después:", limpias.count())

## 5. Agregaciones

In [ ]:
(limpias.groupBy("canal")
 .agg(F.count("*").alias("num_ventas"),
      F.round(F.sum("total")).cast("long").alias("ingresos"),
      F.round(F.avg("total")).cast("long").alias("ticket_promedio"))
 .orderBy(F.desc("ingresos"))
 .show())

## 6. Joins

In [ ]:
completo = (limpias
            .join(productos.select("id_producto", "producto", "categoria"), on="id_producto", how="left")
            .join(clientes.select("id_cliente", "ciudad", "region", "segmento"), on="id_cliente", how="left"))
completo.select("id_venta", "producto", "categoria", "ciudad", "segmento", "total").show(5)

In [ ]:
por_categoria = (completo.groupBy("categoria")
                 .agg(F.round(F.sum("total") / 1e6, 1).alias("ingresos_millones"))
                 .orderBy(F.desc("ingresos_millones")))
por_categoria.show()

## 7. Plan de ejecución: `explain()`
El optimizador **Catalyst** transforma la consulta en un plan físico (Armbrust et al., 2015). En el plan pueden identificarse `BroadcastHashJoin` (unión con tablas pequeñas sin *shuffle*) y `Exchange` (*shuffle*).

In [ ]:
por_categoria.explain()

## 8. De Spark a pandas (solo para resultados pequeños)
`toPandas()` traslada **todos** los datos al *driver*, por lo que solo debe emplearse con resultados agregados.

In [ ]:
import matplotlib.pyplot as plt

pdf = (completo.groupBy("mes").agg(F.sum("total").alias("ingresos"))
       .orderBy("mes").toPandas())

plt.figure(figsize=(9, 4))
plt.bar(pdf["mes"], pdf["ingresos"] / 1e6)
plt.title("Ingresos por mes - 2025 (millones COP)")
plt.xlabel("Mes"); plt.ylabel("Millones")
plt.xticks(range(1, 13))
plt.show()

## Ejercicios propuestos

1. Determinar el **método de pago** más utilizado en cada **canal**.
2. Calcular los ingresos por **región** y **segmento** de cliente.
3. Listar los cinco **clientes** con mayor gasto (nombre, ciudad y total).
4. Identificar los productos con `stock` inferior a 50 y las unidades vendidas de cada uno.

In [ ]:
# Ejercicio 1

In [ ]:
# Ejercicio 2

In [ ]:
# Ejercicio 3

In [ ]:
# Ejercicio 4

In [ ]:
spark.stop()

## Referencias

Armbrust, M., Xin, R. S., Lian, C., Huai, Y., Liu, D., Bradley, J. K., Meng, X., Kaftan, T., Franklin, M. J., Ghodsi, A., & Zaharia, M. (2015). Spark SQL: Relational data processing in Spark. En *Proceedings of the 2015 ACM SIGMOD International Conference on Management of Data* (pp. 1383–1394). ACM. https://doi.org/10.1145/2723372.2742797

The Apache Software Foundation. (2026b). *Spark SQL, DataFrames and Datasets guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/sql-programming-guide.html